# Проект спринта 19. Модуль расчёта батч-признаков для платформы интернет-торговли.
## 1. Установка и импорт библиотек

In [1]:
# Системные библиотеки
import os
from dotenv import load_dotenv
import math
import time
import logging

# Библиотека для доступа к СУБД
from sqlalchemy import create_engine, text

# Библиотеки для вычислений и работы с датасетами
import pandas as pd
import numpy as np

# Для форматирования в HTML
from IPython.display import display, HTML, Markdown

### Определение констант

In [2]:
RUN_DATE = "2025-05-01"

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

## 2. Загрузка параметров и соединение с БД

In [3]:
load_dotenv(".env")

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    os.getenv("DB_USER"),
    os.getenv("DB_PASSWORD"),
    os.getenv("DB_HOST"),
    os.getenv("DB_PORT"),
    os.getenv("DB_NAME")
)

# Создание пула соединений
engine = create_engine(connection_string)

def _get_db_engine():
    return engine

## 3. Загрузка необработанных данных

In [4]:
def show_dataset_info(df, title):
    display(HTML(f"<h2>{title}</h2>"))
    df.info()
    display(df.head())
    display(HTML("<h3>Диапазоны дат</h3>"))
    display(pd.DataFrame({"Начальная дата": df.select_dtypes(include='datetime64').min(),
             "Конечная дата": df.select_dtypes(include='datetime64').max()}))
    display(HTML(f"<h3>Количество уникальных идентификаторов</h3>"))
    display(df.filter(like='_id').nunique())

    display(HTML(f"<h3>Количество полных дубликатов: {df.duplicated().sum()}</h3>"))

def load_all_data():
    db_eng = _get_db_engine()

    df_events = pd.read_sql_table('events', con=db_eng)
    df_sessions = pd.read_sql_table('sessions', con=db_eng)
    df_orders = pd.read_sql_table('orders', con=db_eng)
    df_customers = pd.read_sql_table('customers', con=db_eng)

    return {"df_events": df_events, 
            "df_sessions": df_sessions, 
            "df_orders": df_orders, 
            "df_customers": df_customers}

def load_data_by_run_date(run_date_s: str):
    run_date = pd.Timestamp(run_date_s)
    db_eng = _get_db_engine()

    evt_query = """
        SELECT *
        FROM public.events
        WHERE DATE_TRUNC('day', timestamp) BETWEEN %(run_date)s - 30 AND %(run_date)s - 1
    """
    session_query = """
        SELECT *
        FROM public.sessions
        WHERE DATE_TRUNC('day', start_time) BETWEEN %(run_date)s - 30 AND %(run_date)s - 1
    """
    orders_query = """
        SELECT *
        FROM public.orders
        WHERE DATE_TRUNC('day', order_time) BETWEEN %(run_date)s - 30 AND %(run_date)s - 1
    """
    customers_query = """
        SELECT *
        FROM public.customers
        WHERE DATE_TRUNC('day', signup_date) < %(run_date)s - 30
    """

    df_events = pd.read_sql(evt_query, con=db_eng , params={"run_date": run_date.date()})
    df_sessions = pd.read_sql(session_query, con=db_eng, params={"run_date": run_date.date()})
    df_orders = pd.read_sql(orders_query, con=db_eng, params={"run_date": run_date.date()})
    df_customers = pd.read_sql(customers_query, con=db_eng, params={"run_date": run_date.date()})

    logger.info("Загружены данные за период с %s по %s", run_date - pd.Timedelta(days=30), 
                run_date - pd.Timedelta(days=1))

    return {"df_events": df_events, 
            "df_sessions": df_sessions, 
            "df_orders": df_orders, 
            "df_customers": df_customers}


df_dict = load_all_data()

show_dataset_info(df_dict['df_events'], "Таблица Events")
show_dataset_info(df_dict['df_sessions'], "Таблица Sessions")
show_dataset_info(df_dict['df_orders'], "Таблица Orders")
show_dataset_info(df_dict['df_customers'], "Таблица Customers")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239089 entries, 0 to 239088
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   event_id      239089 non-null  int64         
 1   session_id    239089 non-null  int64         
 2   timestamp     239089 non-null  datetime64[ns]
 3   event_type    239089 non-null  object        
 4   product_id    214208 non-null  float64       
 5   qty           44908 non-null   float64       
 6   cart_size     14229 non-null   float64       
 7   payment       10652 non-null   object        
 8   discount_pct  10652 non-null   float64       
 9   amount_usd    10652 non-null   float64       
dtypes: datetime64[ns](1), float64(5), int64(2), object(2)
memory usage: 18.2+ MB


,event_id,session_id,timestamp,event_type,product_id,qty,cart_size,payment,discount_pct,amount_usd
0,11,2,2025-01-31 21:48:42,page_view,929.0,NaN,NaN,None,NaN,NaN
1,12,2,2025-01-31 21:57:42,page_view,226.0,NaN,NaN,None,NaN,NaN
2,13,2,2025-01-31 21:59:07,add_to_cart,226.0,1.0,NaN,None,NaN,NaN
3,14,2,2025-01-31 22:01:42,page_view,864.0,NaN,NaN,None,NaN,NaN
4,15,2,2025-01-31 22:25:42,page_view,1151.0,NaN,NaN,None,NaN,NaN


,Начальная дата,Конечная дата
timestamp,2024-01-01 01:02:40,2025-11-01 01:50:04


event_id      239089
session_id     37699
product_id      1197
dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37699 entries, 0 to 37698
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   session_id   37699 non-null  int64         
 1   customer_id  37699 non-null  int64         
 2   start_time   37699 non-null  datetime64[ns]
 3   device       37699 non-null  object        
 4   source       37699 non-null  object        
 5   country      37699 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 1.7+ MB


...

order_id       10652
customer_id     8230
dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17846 entries, 0 to 17845
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customer_id       17846 non-null  int64         
 1   name              17846 non-null  object        
 2   email             17846 non-null  object        
 3   country           17846 non-null  object        
 4   age               17846 non-null  int64         
 5   signup_date       17846 non-null  datetime64[ns]
 6   marketing_opt_in  17846 non-null  bool          
dtypes: bool(1), datetime64[ns](1), int64(2), object(3)
memory usage: 854.1+ KB


,customer_id,name,email,country,age,signup_date,marketing_opt_in
0,1,Jennifer Salinas,nicholas59@example.org,JP,71,2020-09-04,True
1,2,Phillip Ramos,christinarubio@example.com,IN,26,2020-04-05,False
2,3,Dawn Fowler,jessica03@example.org,BR,21,2023-08-31,True
3,4,Mario Butler,paula27@example.org,FR,63,2022-06-30,True
4,5,Amber Brown,kevin85@example.net,BR,19,2022-07-22,True


,Начальная дата,Конечная дата
signup_date,2020-01-01,2025-10-31


customer_id    17846
dtype: int64

- Записи в таблицах присутствуют для дат 01.01.2024 по 31.10.2025 включительно.
- Даты для формирования 30-дневных срезов - с 31.01.2024.
- Целевая переменная может быть сформирована для данных до 24.10.2025 включительно.
- Пропуски имеются только в таблице `events`

## 4. Создание батч-признаков

In [5]:
# Предобработка загруженных данных
def preprocess(df_events, df_sessions, df_orders, df_customers):
    df_customers['signup_date'] = pd.to_datetime(df_customers['signup_date'])

    return {"df_events": df_events, 
            "df_sessions": df_sessions, 
            "df_orders": df_orders, 
            "df_customers": df_customers}


# Получение 7- и 30-дневных срезов
def get_slices(run_date_s, df_events, df_sessions, df_orders, df_customers):
    run_date = pd.Timestamp(run_date_s)
    df_dict = {}

    date_from_7 = run_date - pd.Timedelta(days=7)
    date_from_30 = run_date - pd.Timedelta(days=30)

    if df_events.empty:
        raise ValueError(f"Пустой df_events для даты {run_date_s}")
    if date_from_30 < df_events['timestamp'].min().normalize():
        raise ValueError(f"30-дневный срез для этой даты недоступен " + 
                         f"({date_from_30} < {df_events['timestamp'].min().normalize()})")

    # Фильтруем df_customers, чтобы оставить только тех клиентов, которые зарегистрировались 30 и более дней назад
    df_customers_ids = df_customers[df_customers['signup_date'] <= date_from_30][['customer_id']]

    df_dict['events_7'] = (df_events[(df_events['timestamp'] >= date_from_7) &
                                     (df_events['timestamp'] < run_date)]
                           .merge(df_sessions[['session_id', 'customer_id']], on='session_id')
                           .merge(df_customers_ids, on='customer_id', how='inner'))
    
    df_dict['events_30'] = (df_events[(df_events['timestamp'] >= date_from_30) &
                                      (df_events['timestamp'] < run_date)]
                           .merge(df_sessions[['session_id', 'customer_id']], on='session_id')
                           .merge(df_customers_ids, on='customer_id', how='inner'))

    df_dict['sessions_7'] = (df_sessions[(df_sessions['start_time'] >= date_from_7) &
                                        (df_sessions['start_time'] < run_date)]
                            .merge(df_customers_ids, on='customer_id', how='inner'))
    
    df_dict['sessions_30'] = (df_sessions[(df_sessions['start_time'] >= date_from_30) &
                                         (df_sessions['start_time'] < run_date)]
                            .merge(df_customers_ids, on='customer_id', how='inner'))
    
    df_dict['orders_7'] = (df_orders[(df_orders['order_time'] >= date_from_7) &
                                    (df_orders['order_time'] < run_date)]
                            .merge(df_customers_ids, on='customer_id', how='inner'))
    
    df_dict['orders_30'] = (df_orders[(df_orders['order_time'] >= date_from_30) &
                                     (df_orders['order_time'] < run_date)]
                            .merge(df_customers_ids, on='customer_id', how='inner'))

    for df in df_dict.values():
        if df.empty:
            raise ValueError(f"Пустой срез данных для даты {run_date_s}.")
    
    return df_dict

# Создание батч-признаков
def create_batch_features(run_date_s, df_dict):
    df = {}
    run_date = pd.Timestamp(run_date_s)

    df['page_views_7_count'] = (df_dict['events_7'][df_dict['events_7']['event_type'] == 'page_view']
                                .groupby('customer_id')
                                .size()
                                .rename('page_views_7_count'))

    df['page_views_30_count'] = (df_dict['events_30'][df_dict['events_30']['event_type'] == 'page_view']
                                .groupby('customer_id')
                                .size()
                                .rename('page_views_30_count'))

    df['add_to_cart_7_count'] = (df_dict['events_7'][df_dict['events_7']['event_type'] == 'add_to_cart']
                                .groupby('customer_id')
                                .size()
                                .rename('add_to_cart_7_count'))
    
    df['add_to_cart_30_count'] = (df_dict['events_30'][df_dict['events_30']['event_type'] == 'add_to_cart']
                                .groupby('customer_id')
                                .size()
                                .rename('add_to_cart_30_count'))

    df_purchase_count_7 = (df_dict['events_7'][df_dict['events_7']['event_type'] == 'purchase']
                                .groupby('customer_id')
                                .size()
                                .rename('purchase_count_7'))
        
    df_purchase_count_30 = (df_dict['events_30'][df_dict['events_30']['event_type'] == 'purchase']
                                .groupby('customer_id')
                                .size()
                                .rename('purchase_count_30'))

    df['cart_conv_7'] = (df['add_to_cart_7_count'] / df['page_views_7_count']).rename('cart_conv_7')
    df['cart_conv_30'] = (df['add_to_cart_30_count'] / df['page_views_30_count']).rename('cart_conv_30')
    df['purchase_conv_7'] = (df_purchase_count_7 / df['add_to_cart_7_count']).rename('purchase_conv_7')
    df['purchase_conv_30'] = (df_purchase_count_30 / df['add_to_cart_30_count']).rename('purchase_conv_30')
    df['unique_products_7'] = df_dict['events_7'].groupby('customer_id')['product_id'].nunique().rename('unique_products_7')
    df['unique_products_30'] = df_dict['events_30'].groupby('customer_id')['product_id'].nunique().rename('unique_products_30')

    df_session_durations = (df_dict['sessions_30']
                            .merge(df_dict['events_30']
                                      .groupby('session_id')['timestamp']
                                      .max()
                                      .rename('last_event_time'), 
                                    on='session_id', how='left'))
    df_session_durations['session_duration'] = (df_session_durations['last_event_time'] - 
                                                df_session_durations['start_time']).dt.total_seconds()
    df_session_durations['session_duration'] = df_session_durations['session_duration'].fillna(0)
    
    df['mean_session_duration_30'] = (df_session_durations
                                        .groupby('customer_id')['session_duration']
                                        .mean()
                                        .rename('mean_session_duration_30'))
    df['sessions_count_7'] = df_dict['sessions_7'].groupby('customer_id').size().rename('sessions_count_7')
    df['sessions_count_30'] = df_dict['sessions_30'].groupby('customer_id').size().rename('sessions_count_30')

    df_last_purchase = (
        df_dict['events_7']
        .loc[df_dict['events_7']['event_type'] == 'purchase']
        .groupby('customer_id')['timestamp']
        .max()
    )

    df['days_from_last_purchase'] = (
        (run_date.normalize() - df_last_purchase.dt.normalize()).dt.days + 1
    ).rename('days_from_last_purchase')

    df_dict['orders_30']['total_usd'] = df_dict['orders_30']['total_usd'].fillna(0)
    df['orders_count_30'] = df_dict['orders_30'].groupby('customer_id').size().rename('orders_count_30')
    df['total_usd_sum_30'] = df_dict['orders_30'].groupby('customer_id')['total_usd'].sum().rename('total_usd_sum_30')
    df['total_usd_mean_30'] = df_dict['orders_30'].groupby('customer_id')['total_usd'].mean().rename('total_usd_mean_30')

    # Объединение всех признаков в один DataFrame
    batch_features = pd.concat(df.values(), axis=1)

    # Заполнение пропущенных значений из-за деления на ноль или отсутствия покупок
    batch_features.fillna({'days_from_last_purchase': -1}, inplace=True)
    batch_features.fillna(0, inplace=True)

    batch_features['run_date'] = run_date
    
    return batch_features.reset_index()

df_batch = create_batch_features(RUN_DATE, get_slices(RUN_DATE, **preprocess(**load_data_by_run_date(RUN_DATE))))
display(df_batch.head())

,customer_id,page_views_7_count,page_views_30_count,add_to_cart_7_count,add_to_cart_30_count,cart_conv_7,cart_conv_30,purchase_conv_7,purchase_conv_30,unique_products_7,unique_products_30,mean_session_duration_30,sessions_count_7,sessions_count_30,days_from_last_purchase,orders_count_30,total_usd_sum_30,total_usd_mean_30,run_date
0,40,7.0,7.0,2.0,2.0,0.285714,0.285714,0.5,0.5,7.0,7.0,5580.0,1.0,1,8.0,1.0,46.82,46.82,2025-05-01
1,92,3.0,3.0,0.0,0.0,0.000000,0.000000,0.0,0.0,3.0,3.0,4500.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01
2,136,8.0,8.0,3.0,3.0,0.375000,0.375000,0.0,0.0,8.0,8.0,8841.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01
3,148,7.0,7.0,1.0,1.0,0.142857,0.142857,0.0,0.0,7.0,7.0,7950.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01
4,213,4.0,4.0,1.0,1.0,0.250000,0.250000,0.0,0.0,4.0,4.0,4046.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01


## 5. Проверка корректности создания признаков

In [6]:
display(HTML(f"<h3>Количество дубликатов customer_id: {df_batch['customer_id'].duplicated().sum()}</h3>"))
df_batch.info()
display(df_batch.head(10))
df_batch.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1399 entries, 0 to 1398
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype        
---  ------                    --------------  -----        
 0   customer_id               1399 non-null   int64        
 1   page_views_7_count        1399 non-null   float64      
 2   page_views_30_count       1399 non-null   float64      
 3   add_to_cart_7_count       1399 non-null   float64      
 4   add_to_cart_30_count      1399 non-null   float64      
 5   cart_conv_7               1399 non-null   float64      
 6   cart_conv_30              1399 non-null   float64      
 7   purchase_conv_7           1399 non-null   float64      
 8   purchase_conv_30          1399 non-null   float64      
 9   unique_products_7         1399 non-null   float64      
 10  unique_products_30        1399 non-null   float64      
 11  mean_session_duration_30  1399 non-null   float64      
 12  sessions_count_7          1399 non

,customer_id,page_views_7_count,page_views_30_count,add_to_cart_7_count,add_to_cart_30_count,cart_conv_7,cart_conv_30,purchase_conv_7,purchase_conv_30,unique_products_7,unique_products_30,mean_session_duration_30,sessions_count_7,sessions_count_30,days_from_last_purchase,orders_count_30,total_usd_sum_30,total_usd_mean_30,run_date
0,40,7.0,7.0,2.0,2.0,0.285714,0.285714,0.5,0.5,7.0,7.0,5580.0,1.0,1,8.0,1.0,46.82,46.82,2025-05-01
1,92,3.0,3.0,0.0,0.0,0.000000,0.000000,0.0,0.0,3.0,3.0,4500.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01
2,136,8.0,8.0,3.0,3.0,0.375000,0.375000,0.0,0.0,8.0,8.0,8841.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01
3,148,7.0,7.0,1.0,1.0,0.142857,0.142857,0.0,0.0,7.0,7.0,7950.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01
4,213,4.0,4.0,1.0,1.0,0.250000,0.250000,0.0,0.0,4.0,4.0,4046.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01
5,225,1.0,1.0,0.0,0.0,0.000000,0.000000,0.0,0.0,1.0,1.0,540.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01
6,234,7.0,7.0,1.0,1.0,0.142857,0.142857,1.0,1.0,7.0,7.0,9660.0,1.0,1,2.0,1.0,7.26,7.26,2025-05-01
7,240,7.0,7.0,3.0,3.0,0.428571,0.428571,0.0,0.0,7.0,7.0,6180.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01
8,252,1.0,1.0,0.0,0.0,0.000000,0.000000,0.0,0.0,1.0,1.0,1740.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01
9,333,4.0,4.0,1.0,1.0,0.250000,0.250000,0.0,0.0,4.0,4.0,4513.0,1.0,1,-1.0,0.0,0.00,0.00,2025-05-01


,customer_id,page_views_7_count,page_views_30_count,add_to_cart_7_count,add_to_cart_30_count,cart_conv_7,cart_conv_30,purchase_conv_7,purchase_conv_30,unique_products_7,unique_products_30,mean_session_duration_30,sessions_count_7,sessions_count_30,days_from_last_purchase,orders_count_30,total_usd_sum_30,total_usd_mean_30,run_date
count,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399.000000,1399
mean,10017.102931,1.042888,4.791994,0.276626,1.285919,0.061878,0.266509,0.051704,0.209948,1.037884,4.756254,4627.625447,0.233738,1.047891,-0.567548,0.305218,40.064160,39.159203,2025-05-01 00:00:00
min,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,-1.000000,0.000000,0.000000,0.000000,2025-05-01 00:00:00
25%,5066.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,2580.000000,0.000000,1.000000,-1.000000,0.000000,0.000000,0.000000,2025-05-01 00:00:00
50%,10003.000000,0.000000,5.000000,0.000000,1.000000,0.000000,0.250000,0.000000,0.000000,0.000000,5.000000,4500.000000,0.000000,1.000000,-1.000000,0.000000,0.000000,0.000000,2025-05-01 00:00:00
75%,15070.000000,0.000000,7.000000,0.000000,2.000000,0.000000,0.400000,0.000000,0.333333,0.000000,7.000000,6600.000000,0.000000,1.000000,-1.000000,1.000000,25.005000,24.545000,2025-05-01 00:00:00
max,19986.000000,11.000000,16.000000,4.000000,6.000000,1.000000,1.000000,1.000000,1.000000,11.000000,15.000000,11880.000000,2.000000,3.000000,8.000000,2.000000,2680.640000,2680.640000,2025-05-01 00:00:00
std,5772.230780,2.199530,2.582885,0.726538,1.195973,0.167901,0.253008,0.200954,0.360798,2.190660,2.551597,2533.346714,0.430064,0.216935,1.624950,0.474434,119.614726,116.767961,NaN


## 6. Проверка корректности работы DAG
- В папке должен лежать файл с результатом работы DAG и именем `<RUN_DATE>.parquet`

In [7]:
FILE_NAME = f"{RUN_DATE}.parquet"

try:
    if os.path.exists(FILE_NAME):
        df_loaded = pd.read_parquet(FILE_NAME)
    else:
        print(f"Файл {FILE_NAME} не найден")
except:
    print("Ошибка загрузки датасета")
    raise

display(HTML("<h3>Количество отличий между результатом работы DAG и результатом тетрадки:</h3>"))
print((df_batch != df_loaded).sum())

customer_id                 0
page_views_7_count          0
page_views_30_count         0
add_to_cart_7_count         0
add_to_cart_30_count        0
cart_conv_7                 0
cart_conv_30                0
purchase_conv_7             0
purchase_conv_30            0
unique_products_7           0
unique_products_30          0
mean_session_duration_30    0
sessions_count_7            0
sessions_count_30           0
days_from_last_purchase     0
orders_count_30             0
total_usd_sum_30            0
total_usd_mean_30           0
run_date                    0
dtype: int64


- Датасет, полученный в результате работы DAG, полностью совпадает с результатом работы кода в тетрадке.